# 501 — RNAi Associations

## Objective

Evaluate computational associations between the three frozen Phase 4
cross-system transcriptomic consensus programs and the historical combined
DEMETER2 v5 RNAi dependency profiles in the frozen model cohort established by
notebook 107.

RNAi is treated as a methodologically distinct, complementary functional-genomics
layer relative to the DepMap Public 24Q4 CRISPR analysis completed in notebook
500. RNAi and CRISPR evidence remain analytically separate in this notebook and
are not naively pooled.

## Frozen upstream inputs

Notebook 501 consumes only frozen upstream representations:

- the three Phase 4 consensus-program identities, orientations, and cell-line
  scores (`CONSENSUS_TX_01`, `CONSENSUS_TX_02`, and `CONSENSUS_TX_03`);
- the notebook-107 RNAi dependency handoff restricted to the 443 frozen Phase 3
  models with deterministic RNAi mapping;
- the notebook-107 RNAi model, gene-coverage, and screen-source metadata;
- frozen cell-line lineage annotations.

Notebook 501 does not refit, reweight, reorient, rename, rescue, exclude, or
otherwise redefine any frozen consensus program.

## Prespecified RNAi gene universe

The primary RNAi gene universe is restricted to single-gene DEMETER2 targets.
Composite DEMETER2 targets are retained in the audit trail but excluded from
primary gene-level association testing because they do not provide an
unambiguous one-gene analytical unit.

Primary gene eligibility requires:

- at least 75% observed coverage across the 443-model RNAi cohort;
- at least two distinct observed dependency values; and
- non-zero dependency-score variance.

Primary-model estimability is assessed subsequently for each eligible
gene × frozen-program hypothesis after applying the uniform lineage-support
rule. Estimability is therefore a hypothesis-level numerical criterion rather
than a gene-level biological eligibility rule.

A more restrictive 90% coverage rule is reserved for prespecified sensitivity
analysis and does not redefine the primary hypothesis family.

## Primary association framework

For each eligible RNAi gene and each frozen consensus program, the primary model
is:

`DEMETER2 dependency score ~ consensus program score + lineage fixed effects`

Primary inference uses heteroskedasticity-consistent HC3 standard errors.
Lineage is the mandatory adjustment variable, and the frozen consensus-program
score is the predictor of interest. For a given gene × program hypothesis,
lineages represented by fewer than two observed models do not contribute to the
primary fixed-effect estimate; this is applied uniformly as an estimability rule.

Lower DEMETER2 dependency scores represent stronger dependency. Therefore, a
negative consensus-program coefficient indicates that higher program activity is
associated with stronger RNAi dependency and may be described as a
putative-vulnerability-oriented association.

Multiple testing is controlled globally across the complete frozen
program × eligible-gene hypothesis family using the Benjamini–Hochberg
false-discovery-rate procedure at `alpha = 0.05`.

## Prespecified sensitivity and secondary characterization

Two sensitivity analyses are defined without replacing the primary result set:

- a restrictive 90% gene-coverage sensitivity, with FDR recalculated over the
  smaller prespecified hypothesis family; and
- a source-adjusted sensitivity model that adds the frozen RNAi source pattern to
  lineage adjustment.

After the primary FDR-significant evidence set is frozen, pooled Spearman and
within-lineage (`n >= 15`) analyses are used only for result-conditioned
descriptive characterization. Their procedures are prespecified, but they are not
independent validation layers, formal heterogeneity tests, or secondary
significance gates. Directional consistency and lineage heterogeneity remain
continuous contextual evidence dimensions.

## Screen-source heterogeneity

The combined DEMETER2 resource integrates Achilles, DRIVE, and Marcotte RNAi
screens. Their heterogeneous model coverage and source composition are preserved
explicitly.

Screen-source composition is not included in the primary model. The prespecified
sensitivity model additionally adjusts for the frozen RNAi source pattern:

`DEMETER2 dependency score ~ consensus program score + lineage + source pattern`

This sensitivity analysis evaluates residual source-associated heterogeneity
without redefining the primary association family.

## Confounding boundary

No frozen, provenance-supported cell-line proliferation representation is
available for this analysis. No post-hoc proliferation or technical proxy is
introduced; residual proliferation-related confounding is retained as an explicit
limitation. Residual technical and biological confounding may also remain despite
lineage and source-pattern characterization.

## Analytical boundary

Notebook 501 is a functional-vulnerability characterization of frozen programs.

It will not:

- use CRISPR results to select RNAi genes, models, or programs;
- combine CRISPR and RNAi dependency scores, coefficients, p-values, or q-values;
- create an integrated CRISPR–RNAi vulnerability score or ranking;
- use cross-platform concordance as a secondary significance gate;
- introduce post-hoc proliferation or technical proxies;
- perform pharmacogenomic, perturbational, XAI, or compound-prioritization
  analyses;
- redefine any Phase 4 consensus representation.

Formal CRISPR–RNAi correspondence and integrated vulnerability mapping are
deferred to notebook 502, where exact gene identifiers and prespecified
directional categories can be used without retrospectively redefining either
platform-specific analysis.

Negative, heterogeneous, lineage-specific, cross-platform discordant, or
non-recoverable results are valid outputs. Associations are computational
evidence for putative functional vulnerabilities and do not establish causal
dependencies, validated targets, biological mechanisms, or therapeutic efficacy.


In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input paths
# =============================================================================

CONSENSUS_CELLLINE_SCORES_PATH = (
    Paths.consensus_programs
    / "401_consensus_cellline_scores.parquet"
)

RNAI_DEPENDENCY_HANDOFF_PATH = (
    Paths.dependencies
    / "107_rnai_dependency_handoff.parquet"
)

RNAI_FROZEN_COHORT_PATH = (
    Paths.dependencies
    / "107_rnai_frozen_model_cohort.csv"
)

RNAI_GENE_COVERAGE_PATH = (
    Paths.dependencies
    / "107_rnai_gene_coverage.csv"
)

RNAI_MODEL_COVERAGE_PATH = (
    Paths.dependencies
    / "107_rnai_model_coverage.csv"
)

In [3]:
# =============================================================================
# Load frozen upstream inputs
# =============================================================================

consensus_cellline_scores = pd.read_parquet(
    CONSENSUS_CELLLINE_SCORES_PATH
)

rnai_dependency_handoff = pd.read_parquet(
    RNAI_DEPENDENCY_HANDOFF_PATH
)

rnai_frozen_cohort = pd.read_csv(
    RNAI_FROZEN_COHORT_PATH
)

rnai_gene_coverage = pd.read_csv(
    RNAI_GENE_COVERAGE_PATH
)

rnai_model_coverage = pd.read_csv(
    RNAI_MODEL_COVERAGE_PATH
)

In [4]:
# =============================================================================
# Prespecified analysis parameters
# =============================================================================

PRIMARY_COVERAGE_FRACTION = 0.75
SENSITIVITY_COVERAGE_FRACTION = 0.90

MIN_UNIQUE_DEPENDENCY_VALUES = 2
PRIMARY_MIN_LINEAGE_N = 2
WITHIN_LINEAGE_MIN_N = 15

PRIMARY_FDR_ALPHA = 0.05

PRIMARY_MIN_OBSERVED_MODELS = int(
    np.ceil(PRIMARY_COVERAGE_FRACTION * len(rnai_frozen_cohort))
)

SENSITIVITY_MIN_OBSERVED_MODELS = int(
    np.ceil(SENSITIVITY_COVERAGE_FRACTION * len(rnai_frozen_cohort))
)

PRIMARY_FDR_METHOD = "fdr_bh"
PRIMARY_COV_TYPE = "HC3"


FROZEN_CONSENSUS_PROGRAM_IDS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]


In [5]:
# =============================================================================
# Construct RNAi analysis model table
# =============================================================================


missing_consensus_programs = sorted(
    set(FROZEN_CONSENSUS_PROGRAM_IDS)
    - set(consensus_cellline_scores.columns)
)

assert not missing_consensus_programs, (
    f"Missing frozen consensus programs: {missing_consensus_programs}"
)

consensus_program_columns = FROZEN_CONSENSUS_PROGRAM_IDS.copy()

rnai_analysis_models = (
    rnai_model_coverage[
        [
            "ModelID",
            "OncotreeLineage",
            "source_pattern",
        ]
    ]
    .merge(
        consensus_cellline_scores[
            ["ModelID", *consensus_program_columns]
        ],
        on="ModelID",
        how="left",
        validate="one_to_one",
    )
)

In [6]:
# =============================================================================
# Characterize RNAi gene informativeness
# =============================================================================

rnai_gene_informativeness = pd.DataFrame(
    {
        "gene_label": rnai_dependency_handoff.index,
        "dependency_sd": rnai_dependency_handoff.std(axis=1).to_numpy(),
        "n_unique": rnai_dependency_handoff.nunique(
            axis=1,
            dropna=True,
        ).to_numpy(),
    }
)

rnai_gene_characterization = rnai_gene_coverage.merge(
    rnai_gene_informativeness,
    on="gene_label",
    how="left",
    validate="one_to_one",
)

In [7]:
# =============================================================================
# Freeze RNAi gene eligibility
# =============================================================================

rnai_gene_characterization["primary_gene_eligible"] = (
    ~rnai_gene_characterization["is_composite"]
    & (
        rnai_gene_characterization["observed_models"]
        >= PRIMARY_MIN_OBSERVED_MODELS
    )
    & (
        rnai_gene_characterization["n_unique"]
        >= MIN_UNIQUE_DEPENDENCY_VALUES
    )
    & (rnai_gene_characterization["dependency_sd"] > 0)
)

rnai_gene_characterization["coverage90_gene_eligible"] = (
    rnai_gene_characterization["primary_gene_eligible"]
    & (
        rnai_gene_characterization["observed_models"]
        >= SENSITIVITY_MIN_OBSERVED_MODELS
    )
)

In [8]:
# =============================================================================
# Summarize RNAi gene eligibility
# =============================================================================

rnai_gene_eligibility_summary = pd.Series(
    {
        "total_targets": len(rnai_gene_characterization),
        "single_gene_targets": (
            ~rnai_gene_characterization["is_composite"]
        ).sum(),
        "composite_targets": (
            rnai_gene_characterization["is_composite"]
        ).sum(),
        "below_primary_coverage": (
            rnai_gene_characterization["observed_models"]
            < PRIMARY_MIN_OBSERVED_MODELS
        ).sum(),
        "insufficient_unique_values": (
            rnai_gene_characterization["n_unique"]
            < MIN_UNIQUE_DEPENDENCY_VALUES
        ).sum(),
        "zero_variance": (
            rnai_gene_characterization["dependency_sd"] == 0
        ).sum(),
        "primary_gene_eligible": (
            rnai_gene_characterization["primary_gene_eligible"]
        ).sum(),
        "coverage90_gene_eligible": (
            rnai_gene_characterization["coverage90_gene_eligible"]
        ).sum(),
    },
    name="n_targets",
).to_frame()

print(
    "Primary minimum observed models:",
    PRIMARY_MIN_OBSERVED_MODELS,
)
print(
    "90% sensitivity minimum observed models:",
    SENSITIVITY_MIN_OBSERVED_MODELS,
)

rnai_gene_eligibility_summary

Primary minimum observed models: 333
90% sensitivity minimum observed models: 399


,n_targets
total_targets,17309
single_gene_targets,17023
composite_targets,286
below_primary_coverage,4582
insufficient_unique_values,0
zero_variance,0
primary_gene_eligible,12598
coverage90_gene_eligible,6093


In [9]:
# =============================================================================
# Freeze primary analysis universes
# =============================================================================

consensus_program_ids = consensus_program_columns.copy()

eligible_rnai_genes = (
    rnai_gene_characterization.loc[
        rnai_gene_characterization["primary_gene_eligible"],
        "gene_label",
    ]
    .tolist()
)

coverage90_rnai_genes = (
    rnai_gene_characterization.loc[
        rnai_gene_characterization["coverage90_gene_eligible"],
        "gene_label",
    ]
    .tolist()
)

PRIMARY_HYPOTHESIS_COUNT = (
    len(consensus_program_ids)
    * len(eligible_rnai_genes)
)

print("Consensus programs:", len(consensus_program_ids))
print("Primary eligible RNAi genes:", len(eligible_rnai_genes))
print("90% sensitivity RNAi genes:", len(coverage90_rnai_genes))
print("Prespecified primary hypothesis family:", PRIMARY_HYPOTHESIS_COUNT)

Consensus programs: 3
Primary eligible RNAi genes: 12598
90% sensitivity RNAi genes: 6093
Prespecified primary hypothesis family: 37794


In [10]:
# =============================================================================
# Align primary analysis inputs
# =============================================================================

rnai_analysis_models = (
    rnai_analysis_models
    .set_index("ModelID")
    .loc[rnai_dependency_handoff.columns]
)

primary_rnai_dependency = rnai_dependency_handoff.loc[
    eligible_rnai_genes
].copy()

In [11]:
# =============================================================================
# Define primary estimability assessment
# =============================================================================

def assess_primary_estimability(
    dependency_scores,
    program_scores,
    lineage,
):
    analysis_data = pd.DataFrame(
        {
            "dependency_score": dependency_scores,
            "program_score": program_scores,
            "lineage": lineage,
        }
    ).dropna()

    lineage_counts = analysis_data["lineage"].value_counts()
    supported_lineages = lineage_counts[
        lineage_counts >= PRIMARY_MIN_LINEAGE_N
    ].index

    analysis_data = analysis_data[
        analysis_data["lineage"].isin(supported_lineages)
    ]

    if analysis_data.empty:
        return 0, 0, False

    lineage_design = pd.get_dummies(
        analysis_data["lineage"],
        drop_first=True,
        dtype=float,
    )

    design_matrix = pd.concat(
        [
            analysis_data[["program_score"]].astype(float),
            lineage_design,
        ],
        axis=1,
    )

    design_matrix = sm.add_constant(
        design_matrix,
        has_constant="add",
    )

    design_rank = np.linalg.matrix_rank(
        design_matrix.to_numpy()
    )

    is_estimable = (
        design_rank == design_matrix.shape[1]
        and len(analysis_data) > design_matrix.shape[1]
    )

    return (
        len(analysis_data),
        len(supported_lineages),
        is_estimable,
    )

In [12]:
# =============================================================================
# Assess primary hypothesis estimability
# =============================================================================

primary_estimability_records = []

lineage = rnai_analysis_models["OncotreeLineage"]

for gene_label, dependency_scores in primary_rnai_dependency.iterrows():
    for program_id in consensus_program_ids:
        n_models, n_lineages, is_estimable = assess_primary_estimability(
            dependency_scores=dependency_scores,
            program_scores=rnai_analysis_models[program_id],
            lineage=lineage,
        )

        primary_estimability_records.append(
            {
                "gene_label": gene_label,
                "consensus_program_id": program_id,
                "n_models": n_models,
                "n_lineages": n_lineages,
                "primary_estimable": is_estimable,
            }
        )

primary_estimability = pd.DataFrame(
    primary_estimability_records
)

print("Candidate hypotheses:", len(primary_estimability))
print(
    "Estimable hypotheses:",
    primary_estimability["primary_estimable"].sum(),
)
print(
    "Non-estimable hypotheses:",
    (~primary_estimability["primary_estimable"]).sum(),
)

Candidate hypotheses: 37794
Estimable hypotheses: 37794
Non-estimable hypotheses: 0


In [13]:
# =============================================================================
# Freeze primary tested hypothesis family
# =============================================================================

primary_hypothesis_family = (
    primary_estimability.loc[
        primary_estimability["primary_estimable"]
    ]
    .reset_index(drop=True)
    .copy()
)

PRIMARY_TEST_COUNT = len(primary_hypothesis_family)

print(
    "Prespecified candidate hypotheses:",
    PRIMARY_HYPOTHESIS_COUNT,
)
print(
    "Frozen primary tested hypotheses:",
    PRIMARY_TEST_COUNT,
)

Prespecified candidate hypotheses: 37794
Frozen primary tested hypotheses: 37794


In [14]:
# =============================================================================
# Define primary RNAi association model
# =============================================================================

def fit_lineage_adjusted_rnai_association(
    dependency_score,
    program_score,
    lineage,
):
    analysis_data = pd.DataFrame(
        {
            "dependency_score": dependency_score,
            "program_score": program_score,
            "lineage": lineage,
        }
    ).dropna()

    lineage_counts = analysis_data["lineage"].value_counts()
    retained_lineages = lineage_counts.loc[
        lineage_counts >= PRIMARY_MIN_LINEAGE_N
    ].index

    analysis_data = analysis_data.loc[
        analysis_data["lineage"].isin(retained_lineages)
    ]

    lineage_design = pd.get_dummies(
        analysis_data["lineage"],
        drop_first=True,
        dtype=float,
    )

    design = pd.concat(
        [
            analysis_data[["program_score"]].astype(float),
            lineage_design,
        ],
        axis=1,
    )

    design = sm.add_constant(
        design,
        has_constant="add",
    )

    model = sm.OLS(
        analysis_data["dependency_score"].astype(float),
        design,
    ).fit(cov_type=PRIMARY_COV_TYPE)

    confidence_interval = model.conf_int().loc["program_score"]

    return {
        "n_models": int(len(analysis_data)),
        "n_lineages": int(len(retained_lineages)),
        "beta": float(model.params["program_score"]),
        "standard_error": float(model.bse["program_score"]),
        "ci_95_lower": float(confidence_interval.iloc[0]),
        "ci_95_upper": float(confidence_interval.iloc[1]),
        "p_value": float(model.pvalues["program_score"]),
    }

In [15]:
# =============================================================================
# Run primary lineage-adjusted RNAi association screen
# =============================================================================

primary_association_records = []

for program_id in consensus_program_ids:
    program_score = rnai_analysis_models[program_id]

    program_genes = primary_hypothesis_family.loc[
        primary_hypothesis_family["consensus_program_id"] == program_id,
        "gene_label",
    ]

    for gene_label in program_genes:
        association = fit_lineage_adjusted_rnai_association(
            dependency_score=primary_rnai_dependency.loc[gene_label],
            program_score=program_score,
            lineage=rnai_analysis_models["OncotreeLineage"],
        )

        primary_association_records.append(
            {
                "consensus_program_id": program_id,
                "gene_label": gene_label,
                **association,
            }
        )

    print(
        f"Completed {program_id}: "
        f"{len(program_genes):,} associations"
    )

primary_rnai_associations = pd.DataFrame(
    primary_association_records
)

Completed CONSENSUS_TX_01: 12,598 associations
Completed CONSENSUS_TX_02: 12,598 associations
Completed CONSENSUS_TX_03: 12,598 associations


In [16]:
# =============================================================================
# Check primary association output
# =============================================================================

print(
    "Primary associations:",
    len(primary_rnai_associations),
)
print(
    "Non-finite betas:",
    (~np.isfinite(primary_rnai_associations["beta"])).sum(),
)
print(
    "Non-finite standard errors:",
    (
        ~np.isfinite(
            primary_rnai_associations["standard_error"]
        )
    ).sum(),
)
print(
    "Non-finite p-values:",
    (~np.isfinite(primary_rnai_associations["p_value"])).sum(),
)

Primary associations: 37794
Non-finite betas: 0
Non-finite standard errors: 0
Non-finite p-values: 0


In [17]:
# =============================================================================
# Apply global FDR correction
# =============================================================================

primary_fdr_reject, primary_fdr_q_values, _, _ = multipletests(
    primary_rnai_associations["p_value"],
    alpha=PRIMARY_FDR_ALPHA,
    method=PRIMARY_FDR_METHOD,
)

primary_rnai_associations["fdr_q_value"] = primary_fdr_q_values
primary_rnai_associations["fdr_significant"] = primary_fdr_reject

print(
    "Global FDR-significant associations:",
    primary_rnai_associations["fdr_significant"].sum(),
)

Global FDR-significant associations: 1686


In [18]:
# =============================================================================
# Annotate primary RNAi effect direction
# =============================================================================

primary_rnai_associations["effect_direction"] = np.where(
    primary_rnai_associations["beta"] < 0,
    "higher_program_stronger_dependency",
    "higher_program_weaker_dependency",
)

primary_rnai_associations["absolute_beta"] = (
    primary_rnai_associations["beta"].abs()
)

In [19]:
# =============================================================================
# Freeze primary significant RNAi evidence
# =============================================================================

primary_significant_evidence = (
    primary_rnai_associations.loc[
        primary_rnai_associations["fdr_significant"]
    ]
    .reset_index(drop=True)
    .copy()
)

print(
    "Primary FDR-significant RNAi associations:",
    len(primary_significant_evidence),
)

Primary FDR-significant RNAi associations: 1686


In [20]:
# =============================================================================
# Summarize primary significant RNAi evidence
# =============================================================================

primary_significant_summary = (
    primary_significant_evidence
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("gene_label", "size"),
        median_beta=("beta", "median"),
        median_absolute_beta=("absolute_beta", "median"),
        median_fdr_q_value=("fdr_q_value", "median"),
    )
    .reset_index()
)

primary_significant_summary


,consensus_program_id,effect_direction,n_associations,median_beta,median_absolute_beta,median_fdr_q_value
0,CONSENSUS_TX_01,higher_program_stronger_dependency,285,-0.127717,0.127717,0.023900
1,CONSENSUS_TX_01,higher_program_weaker_dependency,191,0.123208,0.123208,0.020886
2,CONSENSUS_TX_02,higher_program_stronger_dependency,263,-0.043863,0.043863,0.020271
3,CONSENSUS_TX_02,higher_program_weaker_dependency,330,0.047162,0.047162,0.021018
4,CONSENSUS_TX_03,higher_program_stronger_dependency,163,-0.043656,0.043656,0.022008
5,CONSENSUS_TX_03,higher_program_weaker_dependency,454,0.049372,0.049372,0.021101


In [21]:
# =============================================================================
# Freeze RNAi putative-vulnerability associations
# =============================================================================

rnai_putative_vulnerability_associations = (
    primary_significant_evidence.loc[
        primary_significant_evidence["effect_direction"]
        == "higher_program_stronger_dependency"
    ]
    .reset_index(drop=True)
    .copy()
)

print(
    "Putative-vulnerability-oriented RNAi associations:",
    len(rnai_putative_vulnerability_associations),
)

Putative-vulnerability-oriented RNAi associations: 711


In [22]:
# =============================================================================
# Characterize putative-vulnerability program recurrence
# =============================================================================

putative_vulnerability_gene_program_count = (
    rnai_putative_vulnerability_associations
    .groupby("gene_label")["consensus_program_id"]
    .nunique()
)

print(
    "Unique putative-vulnerability RNAi genes:",
    putative_vulnerability_gene_program_count.size,
)
print(
    "Genes associated with one program:",
    (putative_vulnerability_gene_program_count == 1).sum(),
)
print(
    "Genes associated with two programs:",
    (putative_vulnerability_gene_program_count == 2).sum(),
)
print(
    "Genes associated with all three programs:",
    (putative_vulnerability_gene_program_count == 3).sum(),
)

Unique putative-vulnerability RNAi genes: 658
Genes associated with one program: 606
Genes associated with two programs: 51
Genes associated with all three programs: 1


In [23]:
# =============================================================================
# Summarize RNAi putative-vulnerability associations
# =============================================================================

putative_vulnerability_summary = (
    rnai_putative_vulnerability_associations
    .groupby("consensus_program_id")
    .agg(
        n_associations=("gene_label", "size"),
        median_beta=("beta", "median"),
        beta_q25=("beta", lambda x: x.quantile(0.25)),
        beta_q75=("beta", lambda x: x.quantile(0.75)),
        median_fdr_q_value=("fdr_q_value", "median"),
    )
    .reset_index()
)

putative_vulnerability_summary

,consensus_program_id,n_associations,median_beta,beta_q25,beta_q75,median_fdr_q_value
0,CONSENSUS_TX_01,285,-0.127717,-0.155892,-0.106077,0.023900
1,CONSENSUS_TX_02,263,-0.043863,-0.056496,-0.036666,0.020271
2,CONSENSUS_TX_03,163,-0.043656,-0.052125,-0.036641,0.022008


## Prespecified 90% coverage sensitivity

The restrictive 90% coverage analysis changes the eligible gene universe but
does not alter the observed model values or fitted primary regression for
gene × program hypotheses retained from the primary screen. Primary OLS
coefficients and p-values are therefore reused for those retained hypotheses,
while Benjamini–Hochberg correction is recalculated over the smaller
prespecified 90%-coverage hypothesis family.

This is a coverage and multiplicity sensitivity analysis, not an independent
replication layer. Associations outside the 90% coverage universe are not
classified as sensitivity failures.


In [24]:
# =============================================================================
# Apply 90% coverage sensitivity
# =============================================================================

coverage90_rnai_associations = (
    primary_rnai_associations.loc[
        primary_rnai_associations["gene_label"].isin(
            coverage90_rnai_genes
        )
    ]
    .reset_index(drop=True)
    .copy()
)

coverage90_fdr_reject, coverage90_fdr_q_values, _, _ = multipletests(
    coverage90_rnai_associations["p_value"],
    alpha=PRIMARY_FDR_ALPHA,
    method=PRIMARY_FDR_METHOD,
)

coverage90_rnai_associations["fdr_q_value_coverage90"] = (
    coverage90_fdr_q_values
)
coverage90_rnai_associations["fdr_significant_coverage90"] = (
    coverage90_fdr_reject
)

print(
    "90% coverage sensitivity hypotheses:",
    len(coverage90_rnai_associations),
)
print(
    "90% coverage FDR-significant associations:",
    coverage90_rnai_associations[
        "fdr_significant_coverage90"
    ].sum(),
)

90% coverage sensitivity hypotheses: 18279
90% coverage FDR-significant associations: 971


In [25]:
# =============================================================================
# Summarize 90% coverage sensitivity agreement
# =============================================================================

primary_significant_in_coverage90 = (
    coverage90_rnai_associations["fdr_significant"]
)

coverage90_significant = (
    coverage90_rnai_associations["fdr_significant_coverage90"]
)

coverage90_agreement_summary = pd.Series(
    {
        "significant_in_both": (
            primary_significant_in_coverage90
            & coverage90_significant
        ).sum(),
        "primary_only_within_coverage90": (
            primary_significant_in_coverage90
            & ~coverage90_significant
        ).sum(),
        "coverage90_only": (
            ~primary_significant_in_coverage90
            & coverage90_significant
        ).sum(),
        "primary_significant_outside_coverage90": (
            primary_rnai_associations["fdr_significant"].sum()
            - primary_significant_in_coverage90.sum()
        ),
    },
    name="n_associations",
).to_frame()

coverage90_agreement_summary

,n_associations
significant_in_both,902
primary_only_within_coverage90,0
coverage90_only,69
primary_significant_outside_coverage90,784


In [26]:
# =============================================================================
# Define source-adjusted RNAi sensitivity model
# =============================================================================

def fit_source_adjusted_rnai_association(
    dependency_score,
    program_score,
    lineage,
    source_pattern,
):
    analysis_data = pd.DataFrame(
        {
            "dependency_score": dependency_score,
            "program_score": program_score,
            "lineage": lineage,
            "source_pattern": source_pattern,
        }
    ).dropna()

    lineage_counts = analysis_data["lineage"].value_counts()
    retained_lineages = lineage_counts.loc[
        lineage_counts >= PRIMARY_MIN_LINEAGE_N
    ].index

    analysis_data = analysis_data.loc[
        analysis_data["lineage"].isin(retained_lineages)
    ]

    lineage_design = pd.get_dummies(
        analysis_data["lineage"],
        drop_first=True,
        dtype=float,
    )

    source_design = pd.get_dummies(
        analysis_data["source_pattern"],
        drop_first=True,
        dtype=float,
    )

    design = pd.concat(
        [
            analysis_data[["program_score"]].astype(float),
            lineage_design,
            source_design,
        ],
        axis=1,
    )

    design = sm.add_constant(
        design,
        has_constant="add",
    )

    design_rank = np.linalg.matrix_rank(
        design.to_numpy()
    )

    if (
        design_rank < design.shape[1]
        or len(analysis_data) <= design.shape[1]
    ):
        return {
            "source_adjusted_estimable": False,
            "n_models": int(len(analysis_data)),
            "n_lineages": int(len(retained_lineages)),
            "n_source_patterns": int(
                analysis_data["source_pattern"].nunique()
            ),
            "beta": np.nan,
            "standard_error": np.nan,
            "ci_95_lower": np.nan,
            "ci_95_upper": np.nan,
            "p_value": np.nan,
        }

    model = sm.OLS(
        analysis_data["dependency_score"].astype(float),
        design,
    ).fit(cov_type=PRIMARY_COV_TYPE)

    confidence_interval = model.conf_int().loc["program_score"]

    inferential_values = np.array(
        [
            model.params["program_score"],
            model.bse["program_score"],
            confidence_interval.iloc[0],
            confidence_interval.iloc[1],
            model.pvalues["program_score"],
        ],
        dtype=float,
    )

    is_estimable = np.isfinite(inferential_values).all()

    return {
        "source_adjusted_estimable": bool(is_estimable),
        "n_models": int(len(analysis_data)),
        "n_lineages": int(len(retained_lineages)),
        "n_source_patterns": int(
            analysis_data["source_pattern"].nunique()
        ),
        "beta": float(inferential_values[0]),
        "standard_error": float(inferential_values[1]),
        "ci_95_lower": float(inferential_values[2]),
        "ci_95_upper": float(inferential_values[3]),
        "p_value": float(inferential_values[4]),
    }

In [27]:
# =============================================================================
# Run source-adjusted RNAi sensitivity
# =============================================================================

source_adjusted_records = []

lineage = rnai_analysis_models["OncotreeLineage"]
source_pattern = rnai_analysis_models["source_pattern"]

for program_id in consensus_program_ids:
    program_score = rnai_analysis_models[program_id]

    program_genes = primary_hypothesis_family.loc[
        primary_hypothesis_family["consensus_program_id"] == program_id,
        "gene_label",
    ]

    for gene_label in program_genes:
        association = fit_source_adjusted_rnai_association(
            dependency_score=primary_rnai_dependency.loc[gene_label],
            program_score=program_score,
            lineage=lineage,
            source_pattern=source_pattern,
        )

        source_adjusted_records.append(
            {
                "consensus_program_id": program_id,
                "gene_label": gene_label,
                **association,
            }
        )

    print(
        f"Completed {program_id}: "
        f"{len(program_genes):,} source-adjusted associations"
    )

source_adjusted_rnai_associations = pd.DataFrame(
    source_adjusted_records
)

Completed CONSENSUS_TX_01: 12,598 source-adjusted associations
Completed CONSENSUS_TX_02: 12,598 source-adjusted associations
Completed CONSENSUS_TX_03: 12,598 source-adjusted associations


In [28]:
# =============================================================================
# Summarize source-adjusted sensitivity estimability
# =============================================================================

source_adjusted_estimability_summary = (
    source_adjusted_rnai_associations
    .groupby("consensus_program_id")
    .agg(
        n_hypotheses=("gene_label", "size"),
        n_estimable=("source_adjusted_estimable", "sum"),
    )
    .reset_index()
)

source_adjusted_estimability_summary["n_non_estimable"] = (
    source_adjusted_estimability_summary["n_hypotheses"]
    - source_adjusted_estimability_summary["n_estimable"]
)

print(
    "Source-adjusted hypotheses:",
    len(source_adjusted_rnai_associations),
)
print(
    "Estimable source-adjusted hypotheses:",
    source_adjusted_rnai_associations[
        "source_adjusted_estimable"
    ].sum(),
)
print(
    "Non-estimable source-adjusted hypotheses:",
    (
        ~source_adjusted_rnai_associations[
            "source_adjusted_estimable"
        ]
    ).sum(),
)

source_adjusted_estimability_summary

Source-adjusted hypotheses: 37794
Estimable source-adjusted hypotheses: 37794
Non-estimable source-adjusted hypotheses: 0


,consensus_program_id,n_hypotheses,n_estimable,n_non_estimable
0,CONSENSUS_TX_01,12598,12598,0
1,CONSENSUS_TX_02,12598,12598,0
2,CONSENSUS_TX_03,12598,12598,0


In [29]:
# =============================================================================
# Apply global FDR correction to source-adjusted sensitivity
# =============================================================================

source_fdr_reject, source_fdr_q_values, _, _ = multipletests(
    source_adjusted_rnai_associations["p_value"],
    alpha=PRIMARY_FDR_ALPHA,
    method=PRIMARY_FDR_METHOD,
)

source_adjusted_rnai_associations[
    "fdr_q_value_source_adjusted"
] = source_fdr_q_values

source_adjusted_rnai_associations[
    "fdr_significant_source_adjusted"
] = source_fdr_reject

print(
    "Source-adjusted FDR-significant associations:",
    source_adjusted_rnai_associations[
        "fdr_significant_source_adjusted"
    ].sum(),
)

Source-adjusted FDR-significant associations: 1507


In [30]:
# =============================================================================
# Compare primary and source-adjusted RNAi associations
# =============================================================================

source_adjusted_comparison = (
    primary_rnai_associations[
        [
            "consensus_program_id",
            "gene_label",
            "beta",
            "fdr_q_value",
            "fdr_significant",
        ]
    ]
    .merge(
        source_adjusted_rnai_associations[
            [
                "consensus_program_id",
                "gene_label",
                "beta",
                "fdr_q_value_source_adjusted",
                "fdr_significant_source_adjusted",
            ]
        ].rename(
            columns={
                "beta": "beta_source_adjusted",
            }
        ),
        on=[
            "consensus_program_id",
            "gene_label",
        ],
        how="inner",
        validate="one_to_one",
    )
)

source_adjusted_comparison["direction_concordant"] = (
    np.sign(source_adjusted_comparison["beta"])
    == np.sign(source_adjusted_comparison["beta_source_adjusted"])
)

source_adjusted_comparison["significance_status"] = np.select(
    [
        (
            source_adjusted_comparison["fdr_significant"]
            & source_adjusted_comparison[
                "fdr_significant_source_adjusted"
            ]
        ),
        (
            source_adjusted_comparison["fdr_significant"]
            & ~source_adjusted_comparison[
                "fdr_significant_source_adjusted"
            ]
        ),
        (
            ~source_adjusted_comparison["fdr_significant"]
            & source_adjusted_comparison[
                "fdr_significant_source_adjusted"
            ]
        ),
    ],
    [
        "significant_in_both",
        "primary_only",
        "source_adjusted_only",
    ],
    default="not_significant_in_either",
)

In [31]:
# =============================================================================
# Summarize source-adjusted sensitivity agreement
# =============================================================================

source_adjusted_significance_summary = (
    source_adjusted_comparison["significance_status"]
    .value_counts()
    .rename_axis("significance_status")
    .rename("n_associations")
    .to_frame()
)

primary_significant_mask = (
    source_adjusted_comparison["fdr_significant"]
)

source_adjusted_direction_summary = pd.Series(
    {
        "direction_concordant_all": (
            source_adjusted_comparison["direction_concordant"]
        ).sum(),
        "direction_discordant_all": (
            ~source_adjusted_comparison["direction_concordant"]
        ).sum(),
        "direction_concordant_primary_significant": (
            source_adjusted_comparison.loc[
                primary_significant_mask,
                "direction_concordant",
            ]
        ).sum(),
        "direction_discordant_primary_significant": (
            ~source_adjusted_comparison.loc[
                primary_significant_mask,
                "direction_concordant",
            ]
        ).sum(),
    },
    name="n_associations",
).to_frame()

source_adjusted_significance_summary

,n_associations
significance_status,
not_significant_in_either,36015
significant_in_both,1414
primary_only,272
source_adjusted_only,93


In [32]:
# =============================================================================
# Display source-adjusted directional agreement
# =============================================================================

source_adjusted_direction_summary

,n_associations
direction_concordant_all,36474
direction_discordant_all,1320
direction_concordant_primary_significant,1686
direction_discordant_primary_significant,0


In [33]:
# =============================================================================
# Annotate primary evidence with source-adjusted sensitivity
# =============================================================================

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        source_adjusted_comparison[
            [
                "consensus_program_id",
                "gene_label",
                "beta_source_adjusted",
                "fdr_q_value_source_adjusted",
                "fdr_significant_source_adjusted",
                "direction_concordant",
            ]
        ],
        on=[
            "consensus_program_id",
            "gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "direction_concordant":
                "source_adjusted_direction_concordant",
        }
    )
)

In [34]:
# =============================================================================
# Define pooled Spearman characterization
# =============================================================================

def compute_pooled_spearman(
    dependency_score,
    program_score,
):
    analysis_data = pd.DataFrame(
        {
            "dependency_score": dependency_score,
            "program_score": program_score,
        }
    ).dropna()

    spearman_rho, _ = spearmanr(
        analysis_data["program_score"],
        analysis_data["dependency_score"],
    )

    return {
        "pooled_n_models": int(len(analysis_data)),
        "pooled_spearman_rho": float(spearman_rho),
    }

In [35]:
# =============================================================================
# Compute pooled Spearman characterization
# =============================================================================

pooled_association_records = []

for row in primary_significant_evidence.itertuples(index=False):
    pooled_association = compute_pooled_spearman(
        dependency_score=primary_rnai_dependency.loc[row.gene_label],
        program_score=rnai_analysis_models[row.consensus_program_id],
    )

    pooled_association_records.append(
        {
            "consensus_program_id": row.consensus_program_id,
            "gene_label": row.gene_label,
            **pooled_association,
        }
    )

pooled_primary_associations = pd.DataFrame(
    pooled_association_records
)

print(
    "Pooled Spearman characterizations:",
    len(pooled_primary_associations),
)

Pooled Spearman characterizations: 1686


In [36]:
# =============================================================================
# Annotate pooled directional consistency
# =============================================================================

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        pooled_primary_associations,
        on=[
            "consensus_program_id",
            "gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

primary_significant_evidence[
    "pooled_adjusted_direction_consistent"
] = (
    np.sign(primary_significant_evidence["pooled_spearman_rho"])
    == np.sign(primary_significant_evidence["beta"])
)

In [37]:
# =============================================================================
# Summarize pooled-adjusted directional consistency
# =============================================================================

pooled_adjusted_direction_summary = (
    primary_significant_evidence
    .groupby("consensus_program_id")
    .agg(
        n_associations=("gene_label", "size"),
        n_direction_consistent=(
            "pooled_adjusted_direction_consistent",
            "sum",
        ),
    )
    .reset_index()
)

pooled_adjusted_direction_summary["n_direction_discordant"] = (
    pooled_adjusted_direction_summary["n_associations"]
    - pooled_adjusted_direction_summary["n_direction_consistent"]
)

pooled_adjusted_direction_summary[
    "direction_consistency_fraction"
] = (
    pooled_adjusted_direction_summary["n_direction_consistent"]
    / pooled_adjusted_direction_summary["n_associations"]
)

print(
    "Direction-consistent associations:",
    primary_significant_evidence[
        "pooled_adjusted_direction_consistent"
    ].sum(),
)
print(
    "Direction-discordant associations:",
    (
        ~primary_significant_evidence[
            "pooled_adjusted_direction_consistent"
        ]
    ).sum(),
)

pooled_adjusted_direction_summary

Direction-consistent associations: 1681
Direction-discordant associations: 5


,consensus_program_id,n_associations,n_direction_consistent,n_direction_discordant,direction_consistency_fraction
0,CONSENSUS_TX_01,476,475,1,0.997899
1,CONSENSUS_TX_02,593,591,2,0.996627
2,CONSENSUS_TX_03,617,615,2,0.996759


In [38]:
# =============================================================================
# Define within-lineage RNAi association model
# =============================================================================

def fit_within_lineage_rnai_association(
    dependency_score,
    program_score,
):
    analysis_data = pd.DataFrame(
        {
            "dependency_score": dependency_score,
            "program_score": program_score,
        }
    ).dropna()

    if len(analysis_data) < WITHIN_LINEAGE_MIN_N:
        return None

    if (
        analysis_data["dependency_score"].nunique() < 2
        or analysis_data["program_score"].nunique() < 2
    ):
        return None

    design = sm.add_constant(
        analysis_data[["program_score"]].astype(float),
        has_constant="add",
    )

    if np.linalg.matrix_rank(design.to_numpy()) < design.shape[1]:
        return None

    model = sm.OLS(
        analysis_data["dependency_score"].astype(float),
        design,
    ).fit(cov_type=PRIMARY_COV_TYPE)

    confidence_interval = model.conf_int().loc["program_score"]

    return {
        "n_models": int(len(analysis_data)),
        "beta": float(model.params["program_score"]),
        "standard_error": float(model.bse["program_score"]),
        "ci_95_lower": float(confidence_interval.iloc[0]),
        "ci_95_upper": float(confidence_interval.iloc[1]),
    }

In [39]:
# =============================================================================
# Run within-lineage RNAi characterization
# =============================================================================

within_lineage_records = []

observed_lineages = (
    rnai_analysis_models["OncotreeLineage"]
    .dropna()
    .unique()
)

for row in primary_significant_evidence.itertuples(index=False):
    dependency_score = primary_rnai_dependency.loc[row.gene_label]
    program_score = rnai_analysis_models[row.consensus_program_id]

    for lineage_name in observed_lineages:
        lineage_model_ids = rnai_analysis_models.index[
            rnai_analysis_models["OncotreeLineage"] == lineage_name
        ]

        association = fit_within_lineage_rnai_association(
            dependency_score=dependency_score.loc[lineage_model_ids],
            program_score=program_score.loc[lineage_model_ids],
        )

        if association is None:
            continue

        within_lineage_records.append(
            {
                "consensus_program_id": row.consensus_program_id,
                "gene_label": row.gene_label,
                "OncotreeLineage": lineage_name,
                **association,
            }
        )

within_lineage_rnai_associations = pd.DataFrame(
    within_lineage_records
)

print(
    "Within-lineage RNAi characterizations:",
    len(within_lineage_rnai_associations),
)
print(
    "Primary associations with at least one evaluable lineage:",
    within_lineage_rnai_associations[
        [
            "consensus_program_id",
            "gene_label",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)

Within-lineage RNAi characterizations: 16929
Primary associations with at least one evaluable lineage: 1686


In [40]:
# =============================================================================
# Summarize within-lineage RNAi heterogeneity
# =============================================================================

within_lineage_annotated = (
    within_lineage_rnai_associations
    .merge(
        primary_significant_evidence[
            [
                "consensus_program_id",
                "gene_label",
                "beta",
            ]
        ].rename(
            columns={"beta": "primary_beta"}
        ),
        on=[
            "consensus_program_id",
            "gene_label",
        ],
        how="left",
        validate="many_to_one",
    )
)

within_lineage_annotated["direction_consistent"] = (
    np.sign(within_lineage_annotated["beta"])
    == np.sign(within_lineage_annotated["primary_beta"])
)

within_lineage_summary = (
    within_lineage_annotated
    .groupby(
        [
            "consensus_program_id",
            "gene_label",
        ]
    )
    .agg(
        n_evaluable_lineages=("OncotreeLineage", "nunique"),
        median_lineage_beta=("beta", "median"),
        lineage_beta_q25=("beta", lambda x: x.quantile(0.25)),
        lineage_beta_q75=("beta", lambda x: x.quantile(0.75)),
        min_lineage_beta=("beta", "min"),
        max_lineage_beta=("beta", "max"),
        n_direction_consistent=(
            "direction_consistent",
            "sum",
        ),
    )
    .reset_index()
)

within_lineage_summary["lineage_beta_iqr"] = (
    within_lineage_summary["lineage_beta_q75"]
    - within_lineage_summary["lineage_beta_q25"]
)

within_lineage_summary["lineage_beta_range"] = (
    within_lineage_summary["max_lineage_beta"]
    - within_lineage_summary["min_lineage_beta"]
)

within_lineage_summary["direction_consistency_fraction"] = (
    within_lineage_summary["n_direction_consistent"]
    / within_lineage_summary["n_evaluable_lineages"]
)

In [41]:
# =============================================================================
# Summarize within-lineage heterogeneity
# =============================================================================

within_lineage_heterogeneity_summary = (
    within_lineage_summary[
        [
            "n_evaluable_lineages",
            "direction_consistency_fraction",
            "lineage_beta_iqr",
            "lineage_beta_range",
        ]
    ]
    .describe(
        percentiles=[0.25, 0.50, 0.75]
    )
    .T
)

within_lineage_heterogeneity_summary

,count,mean,std,min,25%,50%,75%,max
n_evaluable_lineages,1686.0,10.040925,1.043327,8.000000,9.000000,11.000000,11.000000,11.000000
direction_consistency_fraction,1686.0,0.757332,0.120721,0.333333,0.666667,0.777778,0.818182,1.000000
lineage_beta_iqr,1686.0,0.128623,0.088925,0.005691,0.069042,0.103690,0.164042,0.966360
lineage_beta_range,1686.0,0.515034,0.299163,0.079485,0.301330,0.455506,0.656338,2.658635


In [42]:
# =============================================================================
# Annotate primary evidence with lineage heterogeneity
# =============================================================================

within_lineage_evidence = (
    within_lineage_summary
    .rename(
        columns={
            "n_evaluable_lineages":
                "within_lineage_n_evaluable",
            "median_lineage_beta":
                "within_lineage_median_beta",
            "lineage_beta_q25":
                "within_lineage_beta_q25",
            "lineage_beta_q75":
                "within_lineage_beta_q75",
            "min_lineage_beta":
                "within_lineage_min_beta",
            "max_lineage_beta":
                "within_lineage_max_beta",
            "n_direction_consistent":
                "within_lineage_n_direction_consistent",
            "lineage_beta_iqr":
                "within_lineage_beta_iqr",
            "lineage_beta_range":
                "within_lineage_beta_range",
            "direction_consistency_fraction":
                "within_lineage_direction_consistency_fraction",
        }
    )
)

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        within_lineage_evidence,
        on=[
            "consensus_program_id",
            "gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [43]:
# =============================================================================
# Annotate primary evidence with 90% coverage sensitivity
# =============================================================================

coverage90_evidence = (
    coverage90_rnai_associations[
        [
            "consensus_program_id",
            "gene_label",
            "fdr_q_value_coverage90",
            "fdr_significant_coverage90",
        ]
    ]
    .copy()
)

coverage90_evidence["coverage90_evaluable"] = True

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        coverage90_evidence,
        on=[
            "consensus_program_id",
            "gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

primary_significant_evidence["coverage90_evaluable"] = (
    primary_significant_evidence["coverage90_evaluable"]
    .fillna(False)
)

In [44]:
# =============================================================================
# Add frozen RNAi gene annotations
# =============================================================================

rnai_single_gene_annotations = (
    rnai_gene_characterization.loc[
        ~rnai_gene_characterization["is_composite"],
        [
            "gene_label",
            "gene_symbol_group",
            "entrez_id_group",
            "observed_models",
            "coverage_fraction",
        ],
    ]
    .rename(
        columns={
            "gene_symbol_group": "gene_symbol",
            "entrez_id_group": "entrez_id",
            "observed_models": "rnai_observed_models",
            "coverage_fraction": "rnai_coverage_fraction",
        }
    )
    .copy()
)

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        rnai_single_gene_annotations,
        on="gene_label",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Primary significant associations:",
    len(primary_significant_evidence),
)
print(
    "Missing gene symbols:",
    primary_significant_evidence["gene_symbol"].isna().sum(),
)
print(
    "Missing Entrez IDs:",
    primary_significant_evidence["entrez_id"].isna().sum(),
)

Primary significant associations: 1686
Missing gene symbols: 0
Missing Entrez IDs: 0


In [45]:
# =============================================================================
# Refresh putative-vulnerability evidence table
# =============================================================================

rnai_putative_vulnerability_associations = (
    primary_significant_evidence.loc[
        primary_significant_evidence["effect_direction"]
        == "higher_program_stronger_dependency"
    ]
    .reset_index(drop=True)
    .copy()
)

print(
    "Putative-vulnerability-oriented RNAi associations:",
    len(rnai_putative_vulnerability_associations),
)
print(
    "Unique putative-vulnerability RNAi genes:",
    rnai_putative_vulnerability_associations[
        "gene_label"
    ].nunique(),
)

Putative-vulnerability-oriented RNAi associations: 711
Unique putative-vulnerability RNAi genes: 658


In [46]:
# =============================================================================
# Output paths
# =============================================================================

RNAI_MODEL_COHORT_OUTPUT_PATH = (
    Paths.dependencies
    / "501_rnai_model_cohort.csv"
)

RNAI_GENE_ELIGIBILITY_OUTPUT_PATH = (
    Paths.dependencies
    / "501_rnai_gene_eligibility.csv"
)

RNAI_PRIMARY_ASSOCIATIONS_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_primary_associations.csv"
)

RNAI_PRIMARY_SIGNIFICANT_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_primary_significant_evidence.csv"
)

RNAI_PUTATIVE_VULNERABILITIES_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_putative_vulnerability_associations.csv"
)

RNAI_WITHIN_LINEAGE_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_within_lineage_associations.csv"
)

RNAI_SOURCE_ADJUSTED_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_source_adjusted_sensitivity.csv"
)

RNAI_COVERAGE90_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_coverage90_sensitivity.csv"
)

RNAI_METADATA_OUTPUT_PATH = (
    Paths.functional_vulnerabilities
    / "501_rnai_analysis_metadata.json"
)

In [47]:
# =============================================================================
# Prepare RNAi model cohort output
# =============================================================================

rnai_model_cohort_output = (
    rnai_analysis_models
    .rename_axis("ModelID")
    .reset_index()[
        [
            "ModelID",
            "OncotreeLineage",
            "source_pattern",
            *consensus_program_ids,
        ]
    ]
    .copy()
)

print(
    "RNAi analysis models:",
    len(rnai_model_cohort_output),
)
print(
    "Unique ModelIDs:",
    rnai_model_cohort_output["ModelID"].nunique(),
)

RNAi analysis models: 443
Unique ModelIDs: 443


In [48]:
# =============================================================================
# Prepare RNAi gene eligibility output
# =============================================================================

rnai_gene_eligibility_output = (
    rnai_gene_characterization[
        [
            "gene_label",
            "gene_symbol_group",
            "entrez_id_group",
            "n_symbols",
            "n_entrez_ids",
            "is_composite",
            "observed_models",
            "coverage_fraction",
            "dependency_sd",
            "n_unique",
            "primary_gene_eligible",
            "coverage90_gene_eligible",
        ]
    ]
    .copy()
)

print(
    "RNAi targets in eligibility artifact:",
    len(rnai_gene_eligibility_output),
)
print(
    "Primary eligible targets:",
    rnai_gene_eligibility_output[
        "primary_gene_eligible"
    ].sum(),
)
print(
    "90% coverage eligible targets:",
    rnai_gene_eligibility_output[
        "coverage90_gene_eligible"
    ].sum(),
)

RNAi targets in eligibility artifact: 17309
Primary eligible targets: 12598
90% coverage eligible targets: 6093


In [49]:
# =============================================================================
# Prepare primary RNAi associations output
# =============================================================================

rnai_primary_associations_output = (
    primary_rnai_associations
    .merge(
        rnai_single_gene_annotations,
        on="gene_label",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

print(
    "Primary RNAi associations:",
    len(rnai_primary_associations_output),
)
print(
    "Unique tested genes:",
    rnai_primary_associations_output[
        "gene_label"
    ].nunique(),
)
print(
    "Missing gene symbols:",
    rnai_primary_associations_output[
        "gene_symbol"
    ].isna().sum(),
)
print(
    "Missing Entrez IDs:",
    rnai_primary_associations_output[
        "entrez_id"
    ].isna().sum(),
)

Primary RNAi associations: 37794
Unique tested genes: 12598
Missing gene symbols: 0
Missing Entrez IDs: 0


In [50]:
# =============================================================================
# Prepare primary significant RNAi evidence output
# =============================================================================

rnai_primary_significant_output = (
    primary_significant_evidence
    .copy()
)

print(
    "Primary significant RNAi associations:",
    len(rnai_primary_significant_output),
)
print(
    "Unique significant genes:",
    rnai_primary_significant_output[
        "gene_label"
    ].nunique(),
)
print(
    "Putative-vulnerability-oriented associations:",
    (
        rnai_primary_significant_output[
            "effect_direction"
        ]
        == "higher_program_stronger_dependency"
    ).sum(),
)

Primary significant RNAi associations: 1686
Unique significant genes: 1473
Putative-vulnerability-oriented associations: 711


In [51]:
# =============================================================================
# Prepare putative-vulnerability RNAi output
# =============================================================================

rnai_putative_vulnerabilities_output = (
    rnai_putative_vulnerability_associations
    .copy()
)

print(
    "Putative-vulnerability RNAi associations:",
    len(rnai_putative_vulnerabilities_output),
)
print(
    "Unique putative-vulnerability genes:",
    rnai_putative_vulnerabilities_output[
        "gene_label"
    ].nunique(),
)
print(
    "Programs represented:",
    rnai_putative_vulnerabilities_output[
        "consensus_program_id"
    ].nunique(),
)

Putative-vulnerability RNAi associations: 711
Unique putative-vulnerability genes: 658
Programs represented: 3


In [52]:
# =============================================================================
# Prepare within-lineage RNAi associations output
# =============================================================================

rnai_within_lineage_output = (
    within_lineage_annotated
    .merge(
        rnai_single_gene_annotations,
        on="gene_label",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

print(
    "Within-lineage RNAi associations:",
    len(rnai_within_lineage_output),
)
print(
    "Primary associations represented:",
    rnai_within_lineage_output[
        [
            "consensus_program_id",
            "gene_label",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "Lineages represented:",
    rnai_within_lineage_output[
        "OncotreeLineage"
    ].nunique(),
)

Within-lineage RNAi associations: 16929
Primary associations represented: 1686
Lineages represented: 11


In [53]:
# =============================================================================
# Prepare source-adjusted RNAi sensitivity output
# =============================================================================

rnai_source_adjusted_output = (
    source_adjusted_rnai_associations
    .merge(
        rnai_single_gene_annotations,
        on="gene_label",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

print(
    "Source-adjusted RNAi associations:",
    len(rnai_source_adjusted_output),
)
print(
    "Estimable source-adjusted associations:",
    rnai_source_adjusted_output[
        "source_adjusted_estimable"
    ].sum(),
)
print(
    "FDR-significant source-adjusted associations:",
    rnai_source_adjusted_output[
        "fdr_significant_source_adjusted"
    ].sum(),
)

Source-adjusted RNAi associations: 37794
Estimable source-adjusted associations: 37794
FDR-significant source-adjusted associations: 1507


In [54]:
# =============================================================================
# Prepare 90% coverage RNAi sensitivity output
# =============================================================================

rnai_coverage90_output = (
    coverage90_rnai_associations
    .merge(
        rnai_single_gene_annotations,
        on="gene_label",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

print(
    "90% coverage RNAi associations:",
    len(rnai_coverage90_output),
)
print(
    "Unique genes in 90% coverage sensitivity:",
    rnai_coverage90_output[
        "gene_label"
    ].nunique(),
)
print(
    "FDR-significant 90% coverage associations:",
    rnai_coverage90_output[
        "fdr_significant_coverage90"
    ].sum(),
)

90% coverage RNAi associations: 18279
Unique genes in 90% coverage sensitivity: 6093
FDR-significant 90% coverage associations: 971


In [55]:
# =============================================================================
# Prepare RNAi analysis metadata
# =============================================================================

rnai_analysis_metadata = {
    "notebook": "501_rnai_associations",
    "analysis_layer": "RNAi",
    "dependency_measure": "DEMETER2 combined RNAi dependency score",
    "dependency_direction": "lower_score_indicates_stronger_dependency",
    "cohort": {
        "n_models": int(len(rnai_model_cohort_output)),
        "n_consensus_programs": int(len(consensus_program_ids)),
        "consensus_program_ids": consensus_program_ids,
    },
    "gene_universe": {
        "total_targets": int(len(rnai_gene_eligibility_output)),
        "single_gene_targets": int(
            (~rnai_gene_eligibility_output["is_composite"]).sum()
        ),
        "composite_targets_excluded": int(
            rnai_gene_eligibility_output["is_composite"].sum()
        ),
        "primary_coverage_fraction": float(
            PRIMARY_COVERAGE_FRACTION
        ),
        "primary_min_observed_models": int(
            PRIMARY_MIN_OBSERVED_MODELS
        ),
        "primary_eligible_genes": int(
            len(eligible_rnai_genes)
        ),
        "coverage90_fraction": float(
            SENSITIVITY_COVERAGE_FRACTION
        ),
        "coverage90_min_observed_models": int(
            SENSITIVITY_MIN_OBSERVED_MODELS
        ),
        "coverage90_eligible_genes": int(
            len(coverage90_rnai_genes)
        ),
    },
    "primary_model": {
        "formula": (
            "DEMETER2 ~ consensus_program + lineage_fixed_effects"
        ),
        "covariance": PRIMARY_COV_TYPE,
        "minimum_lineage_n": int(PRIMARY_MIN_LINEAGE_N),
        "fdr_method": PRIMARY_FDR_METHOD,
        "fdr_alpha": float(PRIMARY_FDR_ALPHA),
        "hypothesis_family": "global_gene_by_program",
        "n_tests": int(PRIMARY_TEST_COUNT),
    },
    "secondary_characterization": {
        "source_adjusted_model": (
            "DEMETER2 ~ consensus_program "
            "+ lineage_fixed_effects + source_pattern"
        ),
        "pooled_spearman_inferential": False,
        "within_lineage_min_n": int(WITHIN_LINEAGE_MIN_N),
        "within_lineage_inferential": False,
        "proliferation_covariate_used": False,
    },
    "results": {
        "primary_fdr_significant": int(
            len(rnai_primary_significant_output)
        ),
        "putative_vulnerability_associations": int(
            len(rnai_putative_vulnerabilities_output)
        ),
        "putative_vulnerability_unique_genes": int(
            rnai_putative_vulnerabilities_output[
                "gene_label"
            ].nunique()
        ),
        "source_adjusted_fdr_significant": int(
            rnai_source_adjusted_output[
                "fdr_significant_source_adjusted"
            ].sum()
        ),
        "coverage90_fdr_significant": int(
            rnai_coverage90_output[
                "fdr_significant_coverage90"
            ].sum()
        ),
        "within_lineage_characterizations": int(
            len(rnai_within_lineage_output)
        ),
        "within_lineage_lineages": int(
            rnai_within_lineage_output[
                "OncotreeLineage"
            ].nunique()
        ),
    },
    "inputs": {
        "consensus_cellline_scores": project_relative_path(
            CONSENSUS_CELLLINE_SCORES_PATH
        ),
        "rnai_dependency_handoff": project_relative_path(
            RNAI_DEPENDENCY_HANDOFF_PATH
        ),
        "rnai_frozen_cohort": project_relative_path(
            RNAI_FROZEN_COHORT_PATH
        ),
        "rnai_gene_coverage": project_relative_path(
            RNAI_GENE_COVERAGE_PATH
        ),
        "rnai_model_coverage": project_relative_path(
            RNAI_MODEL_COVERAGE_PATH
        ),
    },
    "outputs": {
        "model_cohort": project_relative_path(
            RNAI_MODEL_COHORT_OUTPUT_PATH
        ),
        "gene_eligibility": project_relative_path(
            RNAI_GENE_ELIGIBILITY_OUTPUT_PATH
        ),
        "primary_associations": project_relative_path(
            RNAI_PRIMARY_ASSOCIATIONS_OUTPUT_PATH
        ),
        "primary_significant_evidence": project_relative_path(
            RNAI_PRIMARY_SIGNIFICANT_OUTPUT_PATH
        ),
        "putative_vulnerabilities": project_relative_path(
            RNAI_PUTATIVE_VULNERABILITIES_OUTPUT_PATH
        ),
        "within_lineage_associations": project_relative_path(
            RNAI_WITHIN_LINEAGE_OUTPUT_PATH
        ),
        "source_adjusted_sensitivity": project_relative_path(
            RNAI_SOURCE_ADJUSTED_OUTPUT_PATH
        ),
        "coverage90_sensitivity": project_relative_path(
            RNAI_COVERAGE90_OUTPUT_PATH
        ),
    },
}

In [56]:
# =============================================================================
# Verify prepared RNAi publication outputs
# =============================================================================

publication_checks = {
    "model_cohort_rows": (
        len(rnai_model_cohort_output)
        == len(rnai_frozen_cohort)
    ),
    "model_cohort_unique_ids": (
        rnai_model_cohort_output["ModelID"].nunique()
        == len(rnai_model_cohort_output)
    ),
    "model_cohort_exact_ids": (
        set(rnai_model_cohort_output["ModelID"])
        == set(rnai_frozen_cohort["ModelID"])
    ),
    "gene_eligibility_rows": (
        len(rnai_gene_eligibility_output)
        == len(rnai_gene_characterization)
    ),
    "primary_association_rows": (
        len(rnai_primary_associations_output)
        == PRIMARY_TEST_COUNT
    ),
    "primary_association_unique_keys": (
        ~rnai_primary_associations_output.duplicated(
            [
                "consensus_program_id",
                "gene_label",
            ]
        ).any()
    ),
    "primary_significant_rows": (
        len(rnai_primary_significant_output)
        == primary_rnai_associations[
            "fdr_significant"
        ].sum()
    ),
    "putative_vulnerability_rows": (
        len(rnai_putative_vulnerabilities_output)
        == (
            rnai_primary_significant_output["beta"] < 0
        ).sum()
    ),
    "within_lineage_unique_keys": (
        ~rnai_within_lineage_output.duplicated(
            [
                "consensus_program_id",
                "gene_label",
                "OncotreeLineage",
            ]
        ).any()
    ),
    "source_adjusted_rows": (
        len(rnai_source_adjusted_output)
        == PRIMARY_TEST_COUNT
    ),
    "coverage90_rows": (
        len(rnai_coverage90_output)
        == (
            len(consensus_program_ids)
            * len(coverage90_rnai_genes)
        )
    ),
}

publication_checks = pd.Series(
    publication_checks,
    name="passed",
).to_frame()

assert publication_checks["passed"].all()

json.dumps(rnai_analysis_metadata)

publication_checks

                                 passed
model_cohort_rows                  True
model_cohort_unique_ids            True
model_cohort_exact_ids             True
gene_eligibility_rows              True
primary_association_rows           True
primary_association_unique_keys    True
primary_significant_rows           True
putative_vulnerability_rows        True
within_lineage_unique_keys         True
source_adjusted_rows               True
coverage90_rows                    True

In [57]:
# =============================================================================
# Publish RNAi analysis artifacts
# =============================================================================

for output_path in [
    RNAI_MODEL_COHORT_OUTPUT_PATH,
    RNAI_GENE_ELIGIBILITY_OUTPUT_PATH,
    RNAI_PRIMARY_ASSOCIATIONS_OUTPUT_PATH,
    RNAI_PRIMARY_SIGNIFICANT_OUTPUT_PATH,
    RNAI_PUTATIVE_VULNERABILITIES_OUTPUT_PATH,
    RNAI_WITHIN_LINEAGE_OUTPUT_PATH,
    RNAI_SOURCE_ADJUSTED_OUTPUT_PATH,
    RNAI_COVERAGE90_OUTPUT_PATH,
    RNAI_METADATA_OUTPUT_PATH,
]:
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

rnai_model_cohort_output.to_csv(
    RNAI_MODEL_COHORT_OUTPUT_PATH,
    index=False,
)

rnai_gene_eligibility_output.to_csv(
    RNAI_GENE_ELIGIBILITY_OUTPUT_PATH,
    index=False,
)

rnai_primary_associations_output.to_csv(
    RNAI_PRIMARY_ASSOCIATIONS_OUTPUT_PATH,
    index=False,
)

rnai_primary_significant_output.to_csv(
    RNAI_PRIMARY_SIGNIFICANT_OUTPUT_PATH,
    index=False,
)

rnai_putative_vulnerabilities_output.to_csv(
    RNAI_PUTATIVE_VULNERABILITIES_OUTPUT_PATH,
    index=False,
)

rnai_within_lineage_output.to_csv(
    RNAI_WITHIN_LINEAGE_OUTPUT_PATH,
    index=False,
)

rnai_source_adjusted_output.to_csv(
    RNAI_SOURCE_ADJUSTED_OUTPUT_PATH,
    index=False,
)

rnai_coverage90_output.to_csv(
    RNAI_COVERAGE90_OUTPUT_PATH,
    index=False,
)

with RNAI_METADATA_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        rnai_analysis_metadata,
        handle,
        indent=2,
    )

print("RNAi 501 artifacts published.")

RNAi 501 artifacts published.


In [58]:
# =============================================================================
# Verify published RNAi artifacts
# =============================================================================

published_artifact_paths = [
    RNAI_MODEL_COHORT_OUTPUT_PATH,
    RNAI_GENE_ELIGIBILITY_OUTPUT_PATH,
    RNAI_PRIMARY_ASSOCIATIONS_OUTPUT_PATH,
    RNAI_PRIMARY_SIGNIFICANT_OUTPUT_PATH,
    RNAI_PUTATIVE_VULNERABILITIES_OUTPUT_PATH,
    RNAI_WITHIN_LINEAGE_OUTPUT_PATH,
    RNAI_SOURCE_ADJUSTED_OUTPUT_PATH,
    RNAI_COVERAGE90_OUTPUT_PATH,
    RNAI_METADATA_OUTPUT_PATH,
]

published_artifact_checks = pd.DataFrame(
    {
        "path": [
            project_relative_path(path)
            for path in published_artifact_paths
        ],
        "exists": [
            path.exists()
            for path in published_artifact_paths
        ],
        "size_bytes": [
            path.stat().st_size if path.exists() else 0
            for path in published_artifact_paths
        ],
    }
)

published_artifact_checks["non_empty"] = (
    published_artifact_checks["size_bytes"] > 0
)

assert published_artifact_checks["exists"].all()
assert published_artifact_checks["non_empty"].all()

published_artifact_checks

,path,exists,size_bytes,non_empty
0,data/interim/dependencies/501_rnai_model_cohor...,True,40837,True
1,data/interim/dependencies/501_rnai_gene_eligib...,True,1601331,True
2,data/processed/functional_vulnerabilities/501_...,True,9486536,True
3,data/processed/functional_vulnerabilities/501_...,True,868990,True
4,data/processed/functional_vulnerabilities/501_...,True,370463,True
5,data/processed/functional_vulnerabilities/501_...,True,3016741,True
6,data/processed/functional_vulnerabilities/501_...,True,7682827,True
7,data/processed/functional_vulnerabilities/501_...,True,4937881,True
8,data/processed/functional_vulnerabilities/501_...,True,2946,True


## Analysis summary

Notebook 501 evaluated associations between the three frozen Phase 4
consensus transcriptomic programs and historical DEMETER2 RNAi dependency
profiles in the frozen 443-model RNAi cohort.

The primary analysis was restricted to 12,598 single-gene targets with at
least 75% model coverage and sufficient dependency-score variability.
Composite DEMETER2 targets were excluded without expansion or reassignment.
Primary model estimability was evaluated separately at the gene × program level.

A total of 37,794 gene-by-program hypotheses were evaluated using
lineage-adjusted OLS models with HC3 covariance estimation. Multiple-testing
correction was applied globally across the complete primary hypothesis family
using Benjamini–Hochberg FDR at 0.05.

The primary screen identified 1,686 FDR-significant computational
associations involving 1,473 unique genes. Of these, 711 associations
involving 658 unique genes had negative coefficients, indicating that higher
consensus-program scores were associated with stronger RNAi dependency and
were therefore retained as putative-vulnerability-oriented associations.

Prespecified sensitivities and prespecified procedures for result-conditioned
descriptive characterization were kept separate from primary inference:

- The restrictive 90% coverage sensitivity evaluated 18,279 hypotheses and
  identified 971 FDR-significant associations. Primary regression estimates were
  reused for retained hypotheses because their underlying observations and fitted
  models were unchanged; FDR was recalculated over the smaller coverage-defined
  hypothesis family.
- Additional adjustment for RNAi source pattern identified 1,507
  FDR-significant associations, including 1,414 associations also significant in
  the primary analysis. All 1,686 primary significant associations retained the
  same coefficient direction after source adjustment.
- Pooled Spearman characterization was directionally consistent with the
  lineage-adjusted primary coefficient for 1,681 of 1,686 primary significant
  associations. This pooled analysis is descriptive only.
- Within-lineage characterization generated 16,929 lineage-specific fits
  across 11 evaluable lineages. Directional consistency varied across lineages;
  this heterogeneity is retained as a continuous contextual property rather than
  a binary replication or exclusion criterion.

No frozen, provenance-supported cell-line proliferation covariate was available.
Residual proliferation-related confounding therefore remains an explicit
limitation rather than being addressed with a post-hoc proxy. Residual technical
and biological confounding may also remain despite lineage adjustment and source
characterization.

These analyses do not establish causal dependencies, validated targets, or
clinical predictors. RNAi is treated as a methodologically distinct,
complementary functional-genomics layer relative to CRISPR rather than as an
interchangeable measurement platform. Formal CRISPR–RNAi correspondence and
integrated vulnerability mapping are deferred to notebook 502.
